# 🚦 Real-Time Traffic Signal State Recognition with YOLOv8

**Model:** YOLOv8l &nbsp;|&nbsp; **Classes:** Green · Red · Wait-on · Yellow &nbsp;|&nbsp; **mAP50:** 0.963 &nbsp;|&nbsp; **Speed:** ~4.9 ms/img (TensorRT FP16)

---

## Table of Contents
1. [Dataset Download & Extraction](#1-dataset)
2. [Dependencies](#2-dependencies)
3. [Bounding Box–Guided Cropping & Class Remapping](#3-preprocessing)
4. [Dataset Visualisation (FiftyOne)](#4-visualisation)
5. [YAML Config](#5-yaml)
6. [Training](#6-training)
7. [Training Curves](#7-curves)
8. [Evaluation & Metrics](#8-evaluation)
9. [Save Weights to Drive](#9-save)
10. [TensorRT Export](#10-tensorrt)
11. [Inference](#11-inference)
12. [Demo & Writeup](#12-demo)

---

> **Hardware:** Trained on NVIDIA A100 (Google Colab High-RAM). Reduce `batch` or `imgsz` for smaller GPUs.

## 1. Dataset Download & Extraction <a id='1-dataset'></a>

### Dataset Overview

| Property | Details |
|----------|---------|
| **Source** | Roboflow Universe — Small Traffic Light dataset |
| **Format** | YOLOv8 (YOLO11 compatible) |
| **Original Classes** | 5: Green, off, red, wait_on, yellow |
| **Processed Classes** | 4: Green, red, wait_on, yellow |
| **Challenge** | Small and distant traffic lights in real driving footage |
| **Drive ID** | `1h_joqblzQaPWe4GDTiqfZvLFQgXibj9g` |

The raw dataset contains images from real-world driving scenarios where traffic lights can appear very small (as few as 8×20 pixels). Standard detection pipelines struggle with objects this size because the model sees very few discriminative pixels per object.

In [ ]:
!pip install -q gdown

In [ ]:
import gdown

file_id = "1h_joqblzQaPWe4GDTiqfZvLFQgXibj9g"
output  = "/content/dataset.zip"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

In [ ]:
!unzip -q "/content/dataset.zip" -d /content/Small_Traffic_Light.v1i.yolov11
print("Dataset extracted ✅")

## 2. Dependencies <a id='2-dependencies'></a>

In [ ]:
!pip install -q ultralytics tensorrt fiftyone opencv-python tqdm

In [ ]:
import os
import cv2
import json
import shutil
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from pathlib import Path
from glob import glob
from tqdm.auto import tqdm
from PIL import Image as PILImage
from IPython.display import Image, display
from ultralytics import YOLO

print("All dependencies imported ✅")

## 3. Bounding Box–Guided Cropping & Class Remapping <a id='3-preprocessing'></a>

### Strategy

Instead of traditional fixed-grid tiling, this pipeline uses **bounding box–guided cropping**:

| Approach | Pros | Cons |
|----------|------|------|
| **Grid tiling** | Many background patches → model learns when NOT to detect | Ignores object size; tiny objects still tiny |
| **BBox-guided crops** | Small objects appear large → more discriminative features | No background-only patches → risk of false positives |
| **This project (both)** | High-res crops + original full images → best of both worlds | Slightly larger dataset |

### Cropping Algorithm

1. For each annotated bounding box, expand the region by `CROP_EXPAND × object_size` in all directions.
2. Any other bounding boxes whose centres fall inside the expanded crop are included in the same label file.
3. Boxes already captured by a previous crop are not processed again (no redundant duplicates).
4. The original full image is always copied alongside its crops.

### Class Remapping

The `off` class had only **7 instances** after preprocessing and was **incorrectly labeled** (visually identical to red). Duplicating mislabeled samples would amplify noise. Instead, `off` is merged into `red`:

```
Original:  0=Green  1=off  2=red  3=wait_on  4=yellow
Processed: 0=Green  1=red         2=wait_on  3=yellow
```

In [ ]:
import os
import cv2
import shutil
from tqdm import tqdm

dataset_root = "/content/Small_Traffic_Light.v1i.yolov11/Small Traffic Light.v1i.yolov11"
output_root  = "/content/traffic_light_crops_dataset"

CROP_EXPAND = 9    # expand crop by 9× object size in each direction
MIN_SIZE    = 10   # skip objects smaller than 10px in either dimension
VALID_EXTS  = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# Original class → new class mapping
CLASS_REMAP = {
    0: 0,  # Green   → Green
    1: 1,  # off     → red   (mislabeled)
    2: 1,  # red     → red
    3: 2,  # wait_on → wait_on
    4: 3,  # yellow  → yellow
}
CLASS_NAMES = {0: "Green", 1: "red", 2: "wait_on", 3: "yellow"}

# ── Coordinate helpers ────────────────────────────────────────────────────────
def yolo_to_xyxy(box, img_w, img_h):
    xc, yc, bw, bh = box
    x1 = int((xc - bw / 2) * img_w)
    y1 = int((yc - bh / 2) * img_h)
    x2 = int((xc + bw / 2) * img_w)
    y2 = int((yc + bh / 2) * img_h)
    return x1, y1, x2, y2

def xyxy_to_yolo(box, img_w, img_h):
    x1, y1, x2, y2 = box
    xc = ((x1 + x2) / 2) / img_w
    yc = ((y1 + y2) / 2) / img_h
    bw = (x2 - x1) / img_w
    bh = (y2 - y1) / img_h
    return [xc, yc, bw, bh]

def clip(val, lo, hi):
    return max(lo, min(val, hi))

# ── Label I/O ─────────────────────────────────────────────────────────────────
def read_labels(path):
    """Read YOLO labels and apply CLASS_REMAP."""
    boxes = []
    if not os.path.exists(path):
        return boxes
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cid = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:])
            if cid not in CLASS_REMAP:
                continue
            boxes.append((CLASS_REMAP[cid], [xc, yc, bw, bh]))
    return boxes

def write_labels(path, boxes):
    with open(path, "w") as f:
        for cid, (xc, yc, bw, bh) in boxes:
            f.write(f"{cid} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

# ── Per-split processor ───────────────────────────────────────────────────────
def process_split(split):
    img_dir = os.path.join(dataset_root, split, "images")
    lbl_dir = os.path.join(dataset_root, split, "labels")
    out_img = os.path.join(output_root, split, "images")
    out_lbl = os.path.join(output_root, split, "labels")
    os.makedirs(out_img, exist_ok=True)
    os.makedirs(out_lbl, exist_ok=True)

    images = [f for f in os.listdir(img_dir)
               if os.path.splitext(f)[1].lower() in VALID_EXTS]
    crop_count = 0

    for img_file in tqdm(images, desc=f"[{split}]"):
        stem = os.path.splitext(img_file)[0]
        img  = cv2.imread(os.path.join(img_dir, img_file))
        if img is None:
            continue
        h, w = img.shape[:2]
        boxes = read_labels(os.path.join(lbl_dir, stem + ".txt"))

        # 1. Copy original with remapped labels
        shutil.copy(os.path.join(img_dir, img_file), os.path.join(out_img, img_file))
        write_labels(os.path.join(out_lbl, stem + ".txt"), boxes)

        # 2. Generate BBox-guided crops
        visited = set()
        for i, (cls_i, box_i) in enumerate(boxes):
            if i in visited:
                continue
            x1, y1, x2, y2 = yolo_to_xyxy(box_i, w, h)
            obj_w, obj_h = x2 - x1, y2 - y1
            if obj_w < MIN_SIZE or obj_h < MIN_SIZE:
                continue

            cx1 = clip(x1 - obj_w * CROP_EXPAND, 0, w)
            cy1 = clip(y1 - obj_h * CROP_EXPAND, 0, h)
            cx2 = clip(x2 + obj_w * CROP_EXPAND, 0, w)
            cy2 = clip(y2 + obj_h * CROP_EXPAND, 0, h)
            crop = img[int(cy1):int(cy2), int(cx1):int(cx2)]
            ch, cw = crop.shape[:2]
            if cw < MIN_SIZE or ch < MIN_SIZE:
                continue

            crop_boxes = []
            for j, (cls_j, box_j) in enumerate(boxes):
                bx1, by1, bx2, by2 = yolo_to_xyxy(box_j, w, h)
                bcx, bcy = (bx1 + bx2) / 2, (by1 + by2) / 2
                if cx1 <= bcx <= cx2 and cy1 <= bcy <= cy2:
                    nx1 = clip(bx1 - cx1, 0, cw)
                    ny1 = clip(by1 - cy1, 0, ch)
                    nx2 = clip(bx2 - cx1, 0, cw)
                    ny2 = clip(by2 - cy1, 0, ch)
                    crop_boxes.append((cls_j, xyxy_to_yolo([nx1, ny1, nx2, ny2], cw, ch)))
                    visited.add(j)

            if not crop_boxes:
                continue

            ext  = os.path.splitext(img_file)[1]
            name = f"{stem}_crop{crop_count:04d}{ext}"
            cv2.imwrite(os.path.join(out_img, name), crop)
            write_labels(os.path.join(out_lbl, os.path.splitext(name)[0] + ".txt"), crop_boxes)
            crop_count += 1

    print(f"  [{split}] {crop_count} crops generated.")

# ── Run all splits ────────────────────────────────────────────────────────────
for split in ["train", "valid", "test"]:
    split_path = os.path.join(dataset_root, split, "images")
    if os.path.isdir(split_path):
        process_split(split)
    else:
        print(f"  Split '{split}' not found — skipping.")

print("\n✅ Dataset preparation complete.")

## 4. Dataset Visualisation with FiftyOne <a id='4-visualisation'></a>

FiftyOne provides an interactive browser-based UI for exploring annotations, spotting label errors, and filtering by class or confidence. Below we load both the training and validation splits.

In [ ]:
import fiftyone as fo
import fiftyone.types as fot

classes = ["Green", "red", "wait_on", "yellow"]

train_dataset = fo.Dataset.from_dir(
    dataset_type=fot.YOLOv4Dataset,
    data_path="/content/traffic_light_crops_dataset/train/images",
    labels_path="/content/traffic_light_crops_dataset/train/labels",
    label_field="ground_truth",
    classes=classes,
)
print(f"Train samples : {len(train_dataset)}")

In [ ]:
valid_dataset = fo.Dataset.from_dir(
    dataset_type=fot.YOLOv4Dataset,
    data_path="/content/traffic_light_crops_dataset/valid/images",
    labels_path="/content/traffic_light_crops_dataset/valid/labels",
    label_field="ground_truth",
    classes=classes,
)
print(f"Valid samples : {len(valid_dataset)}")

In [ ]:
# Launch FiftyOne app — copy the URL into a browser tab
session = fo.launch_app(train_dataset, remote=False)
print("FiftyOne App URL:", session.url)

## 5. Dataset YAML Config <a id='5-yaml'></a>

YOLOv8 requires a YAML file specifying dataset paths and class names.

In [ ]:
import yaml

data = {
    'path'  : '/content/traffic_light_crops_dataset',
    'train' : 'train/images',
    'val'   : 'valid/images',
    'names' : {
        0: 'Green',
        1: 'red',
        2: 'wait_on',
        3: 'yellow',
    },
}

yaml_path = 'traffic_light_detection.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print(f"YAML saved to: {yaml_path} ✅")
print(open(yaml_path).read())

## 6. Training <a id='6-training'></a>

### Hyperparameter Rationale

| Hyperparameter | Value | Reason |
|---------------|-------|--------|
| `model` | YOLOv8l | Best accuracy/speed trade-off for detection |
| `imgsz` | 960 | Higher resolution → more pixels per small traffic light |
| `epochs` | 100 | Sufficient for convergence on this dataset size |
| `batch` | 16 | Fits A100 VRAM at imgsz=960 |
| `hsv_h` | 0.01 | Subtle hue shift → handles varying light colour temperature |
| `hsv_s` | 0.70 | Saturation aug → overcast / rainy conditions |
| `hsv_v` | 0.40 | Brightness aug → glare, night, shadows |

In [ ]:
from ultralytics import YOLO

EPOCHS = 100
model  = YOLO('yolov8l.pt')

results = model.train(
    data   = '/content/traffic_light_detection.yaml',
    epochs = EPOCHS,
    imgsz  = 960,
    batch  = 16,
    hsv_h  = 0.01,
    hsv_s  = 0.70,
    hsv_v  = 0.40,
)

print("\n✅ Training complete.")
print("   Best weights: runs/detect/train/weights/best.pt")

## 7. Training Curves <a id='7-curves'></a>

In [ ]:
results_img = plt.imread('runs/detect/train/results.png')
plt.figure(figsize=(16, 10))
plt.imshow(results_img)
plt.axis('off')
plt.title('Training Curves — Loss, Precision, Recall, mAP', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Evaluation & Metrics <a id='8-evaluation'></a>

### Expected Results (after label fix)

| Metric | Before (5 classes, `off` present) | After (4 classes, `off` merged) |
|--------|-----------------------------------|----------------------------------|
| mAP50-95 | 0.436 | **0.542** |
| Precision | — | **0.967** |
| Recall | — | **0.932** |
| mAP50 | — | **0.963** |

In [ ]:
model   = YOLO('/content/runs/detect/train/weights/best.pt')
metrics = model.val(data="/content/traffic_light_detection.yaml", save=True)

print("\n=" * 30)
print("  Overall Metrics")
print("=" * 30)
print(f"  mAP50-95 : {metrics.box.map:.4f}")
print(f"  mAP50    : {metrics.box.map50:.4f}")
print(f"  mAP75    : {metrics.box.map75:.4f}")
print()
print("  Per-Class mAP50-95:")
class_names = ["Green", "red", "wait_on", "yellow"]
for i, ap in enumerate(metrics.box.maps):
    name = class_names[i] if i < len(class_names) else f"class_{i}"
    print(f"    {name:<12} {ap:.4f}")

## 9. Save Best Weights to Google Drive <a id='9-save'></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil

src  = '/content/runs/detect/train/weights/best.pt'
dst  = '/content/drive/MyDrive/yolov8l_960imgsz_TrafficLight_Detect_100ep_best.pt'
shutil.copy(src, dst)

print(f"Model saved to Drive ✅")
print(f"  → {dst}")

## 10. TensorRT Export <a id='10-tensorrt'></a>

Exporting to TensorRT FP16 achieves **~4.9 ms/image** on an A100, enabling deployment in real-time automotive systems.

| Format | Speed | Notes |
|--------|-------|-------|
| PyTorch `.pt` | ~15–25 ms | Standard, portable |
| **TensorRT `.engine`** | **~4.9 ms** | NVIDIA GPU only, maximum throughput |

In [ ]:
# Export to TensorRT FP16
model.export(format="engine", half=True)

# Load the exported TensorRT engine
trt_model = YOLO("/content/runs/detect/train/weights/best.engine")
print("TensorRT engine loaded ✅")

## 11. Inference <a id='11-inference'></a>

Run inference on a video using the TensorRT engine. Confidence threshold set to 0.70 to minimise false positives.

In [ ]:
trt_model.predict(
    source = "/content/inference_traffic_light_video.mp4",
    save   = True,
    conf   = 0.70,
    name   = "out_TrafficLightDemo",
)
print("Inference complete ✅")
print("Output: runs/detect/out_TrafficLightDemo/")

## 12. Demo & Writeup <a id='12-demo'></a>

---

### 🎥 Inference Demo Video

[![Demo Video](https://img.youtube.com/vi/Iu9jzexylSs/maxresdefault.jpg)](https://youtu.be/Iu9jzexylSs)

**▶ [Watch on YouTube — youtu.be/Iu9jzexylSs](https://youtu.be/Iu9jzexylSs)**

---

### 📄 Full Strategy Writeup

The complete technical writeup covering all strategies, decisions, and results is available here:

**[📝 Read the Writeup Document (Google Docs)](https://docs.google.com/document/d/1BYa89EUTyGwMNhOo5IqY0mL8Tp3aesl__1rMTd0Ivbk/edit?usp=sharing)**

---

### Summary of Key Findings

| Finding | Impact |
|---------|--------|
| BBox-guided cropping (vs. grid tiling) | Improved small-object detection significantly |
| Retaining original full images | Reduced false positives, improved generalization |
| Merging mislabeled `off` → `red` | mAP50-95 improved from 0.436 → **0.542** |
| TensorRT FP16 export | Inference speed: ~4.9 ms/image |

> **Core lesson:** Thoughtful data preparation — correcting labels, balancing classes, and using context-aware augmentation — consistently outperforms architectural changes alone.